# Phase 1 — Data Pipeline

This notebook demonstrates the complete data engineering workflow using the production code in `src/fx_forecast`.

In [4]:
import pandas as pd

from fx_forecast.config.settings import settings
from fx_forecast.config.paths import RAW_DATA_DIR, PROCESSED_DATA_DIR
from fx_forecast.data.fetch import DataFetcher
from fx_forecast.data.validate import validate_dataframe
from fx_forecast.data.preprocess import preprocess_dataframe
from fx_forecast.data.pipeline import run_pipeline

## Inspect project settings

In [5]:
print("Currency pairs:", settings.currency_pairs)
print("Start date:", settings.start_date)
print("Interval:", settings.interval)
print("Random seed:", settings.random_seed)

Currency pairs: ['EURUSD=X', 'GBPUSD=X', 'USDJPY=X']
Start date: 2015-01-01
Interval: 1d
Random seed: 42


## Select one currency pair for demonstration

In [6]:
symbol = settings.currency_pairs[0]
print(symbol)

EURUSD=X


## Download raw market data

In [7]:
fetcher = DataFetcher(RAW_DATA_DIR)

raw_df = fetcher.download(
    symbol=symbol,
    start=settings.start_date,
    interval=settings.interval,
)

raw_df.head()

2026-08-04 13:23:04 | INFO     | fx_forecast.data.fetch:download:35 | Downloading EURUSD=X (2015-01-01 → 2026-08-04)
2026-08-04 13:24:09 | SUCCESS  | fx_forecast.data.io:save_dataframe:56 | Saved dataset -> C:\Engineering\02_Projects\fx-forecast-system\data\raw\EURUSD_X.csv
2026-08-04 13:24:10 | SUCCESS  | fx_forecast.data.fetch:download:66 | EURUSD=X: 3015 rows downloaded


,Close,High,Low,Open,Volume
Date,,,,,
2015-01-01,1.209863,1.209863,1.209863,1.209863,0
2015-01-02,1.208941,1.208956,1.201080,1.208868,0
2015-01-05,1.194643,1.197590,1.188909,1.195500,0
2015-01-06,1.193902,1.197000,1.188693,1.193830,0
2015-01-07,1.187536,1.190000,1.180401,1.187479,0


## Explore the downloaded data

In [8]:
raw_df.info()
raw_df.describe().T

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3015 entries, 2015-01-01 to 2026-08-03
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   3015 non-null   float64
 1   High    3015 non-null   float64
 2   Low     3015 non-null   float64
 3   Open    3015 non-null   float64
 4   Volume  3015 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 141.3 KB


,count,mean,std,min,25%,50%,75%,max
Close,3015.0,1.122249,0.051748,0.959619,1.086195,1.120599,1.163196,1.251001
High,3015.0,1.125778,0.051484,0.967006,1.089452,1.124099,1.165902,1.255808
Low,3015.0,1.118679,0.051940,0.954016,1.082784,1.116807,1.160174,1.245051
Open,3015.0,1.122233,0.051742,0.959619,1.086154,1.120498,1.163291,1.251267
Volume,3015.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## Validate the dataset

In [9]:
validated_df = validate_dataframe(raw_df)
validated_df.head()

2026-08-04 13:26:06 | SUCCESS  | fx_forecast.data.validate:validate_dataframe:57 | Dataset validation passed.


,Close,High,Low,Open,Volume
Date,,,,,
2015-01-01,1.209863,1.209863,1.209863,1.209863,0
2015-01-02,1.208941,1.208956,1.201080,1.208868,0
2015-01-05,1.194643,1.197590,1.188909,1.195500,0
2015-01-06,1.193902,1.197000,1.188693,1.193830,0
2015-01-07,1.187536,1.190000,1.180401,1.187479,0


## Preprocess the dataset

In [10]:
processed_df = preprocess_dataframe(validated_df)
processed_df.head()

2026-08-04 13:27:25 | SUCCESS  | fx_forecast.data.preprocess:preprocess_dataframe:59 | Preprocessing complete (3015 → 3015 rows).


,Close,High,Low,Open,Volume
Date,,,,,
2015-01-01,1.209863,1.209863,1.209863,1.209863,0
2015-01-02,1.208941,1.208956,1.201080,1.208868,0
2015-01-05,1.194643,1.197590,1.188909,1.195500,0
2015-01-06,1.193902,1.197000,1.188693,1.193830,0
2015-01-07,1.187536,1.190000,1.180401,1.187479,0


## Execute the complete production pipeline

In [11]:
pipeline_df = run_pipeline(
    symbol=symbol,
    start=settings.start_date,
    interval=settings.interval,
)

pipeline_df.head()

2026-08-04 13:28:24 | INFO     | fx_forecast.data.pipeline:run_pipeline:27 | Starting pipeline for EURUSD=X
2026-08-04 13:28:24 | INFO     | fx_forecast.data.fetch:download:35 | Downloading EURUSD=X (2015-01-01 → 2026-08-04)
2026-08-04 13:28:25 | SUCCESS  | fx_forecast.data.io:save_dataframe:56 | Saved dataset -> C:\Engineering\02_Projects\fx-forecast-system\data\raw\EURUSD_X.csv
2026-08-04 13:28:25 | SUCCESS  | fx_forecast.data.fetch:download:66 | EURUSD=X: 3015 rows downloaded
2026-08-04 13:28:25 | SUCCESS  | fx_forecast.data.validate:validate_dataframe:57 | Dataset validation passed.
2026-08-04 13:28:25 | SUCCESS  | fx_forecast.data.preprocess:preprocess_dataframe:59 | Preprocessing complete (3015 → 3015 rows).
2026-08-04 13:28:26 | SUCCESS  | fx_forecast.data.io:save_dataframe:56 | Saved dataset -> C:\Engineering\02_Projects\fx-forecast-system\data\processed\EURUSD_X.csv
2026-08-04 13:28:26 | SUCCESS  | fx_forecast.data.pipeline:run_pipeline:48 | Pipeline completed for EURUSD=X


,Close,High,Low,Open,Volume
Date,,,,,
2015-01-01,1.209863,1.209863,1.209863,1.209863,0
2015-01-02,1.208941,1.208956,1.201080,1.208868,0
2015-01-05,1.194643,1.197590,1.188909,1.195500,0
2015-01-06,1.193902,1.197000,1.188693,1.193830,0
2015-01-07,1.187536,1.190000,1.180401,1.187479,0


## Verify processed files

In [12]:
list(PROCESSED_DATA_DIR.glob("*.csv"))

[WindowsPath('C:/Engineering/02_Projects/fx-forecast-system/data/processed/EURUSD_X.csv'),
 WindowsPath('C:/Engineering/02_Projects/fx-forecast-system/data/processed/test.csv')]

# Phase 1 Summary

✔ Download historical FX data

✔ Validate data quality

✔ Clean and preprocess data

✔ Save processed dataset

The processed data is now ready for Phase 2 (Feature Engineering).